In [ ]:
import torch
import json
from dotenv import load_dotenv
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

load_dotenv()

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.95)


## Config

In [ ]:
import os

MODEL_PATH      = "./models/qwen2.5-0.5b-instruct"
TRAIN_DATA_PATH = "dataset_output/train.json"
VAL_DATA_PATH   = "dataset_output/validation.json"
OUTPUT_DIR      = "lora_output"
MAX_LENGTH      = 320

with open("schema_prompt.txt") as f:
    SCHEMA = f.read()

## Load tokenizer & model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16,
    device_map="mps" if torch.backends.mps.is_available() else "cpu",
)
model.config.use_cache = False
device = next(model.parameters()).device
print(f"Using device: {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")


## Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,                          # reduced from 8, halves LoRA activation memory
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Load & format dataset

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

def format_sample(sample):
    prompt_messages = [
        {
            "role": "system",
            "content": "You are a SQL expert. Given a database schema and a question, write the correct SQL query."
        },
        {
            "role": "user",
            "content": f"Schema:{SCHEMA}\nQuestion: {sample['natural_language']}"
        },
    ]
    return {
        "prompt":     tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True),
        "completion": sample["sql_query"] + tokenizer.eos_token,
    }

train_dataset = Dataset.from_list(load_json(TRAIN_DATA_PATH)).map(format_sample)
val_dataset   = Dataset.from_list(load_json(VAL_DATA_PATH)).map(format_sample)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print("\nSample prompt (truncated):")
print(train_dataset[0]["prompt"][:200])
print("\nSample completion:")
print(train_dataset[0]["completion"])


## Training arguments

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=False,
    logging_steps=10,
    bf16=False,
    fp16=True,
    report_to="none",
    optim="adafactor",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    max_length=MAX_LENGTH,
)


## Train

In [ ]:
import os
import re

def get_latest_checkpoint(output_dir):
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [
        os.path.join(output_dir, d)
        for d in os.listdir(output_dir)
        if re.match(r"checkpoint-\d+", d)
    ]
    return max(checkpoints, key=lambda p: int(p.split("-")[-1])) if checkpoints else None

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

checkpoint = get_latest_checkpoint(OUTPUT_DIR)
if checkpoint:
    print(f"Resuming from checkpoint: {checkpoint}")
else:
    print("No checkpoint found, starting from scratch.")

trainer.train(resume_from_checkpoint=checkpoint)


## Save LoRA adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to: {OUTPUT_DIR}")